In [1]:
# ============================================================================
# Kaggle-Optimized: Bottleneck Formation in Deep Networks (CIFAR-10 Edition)
# Version 6 - Full Code with CIFAR-10, Time Management, and 14 Plots
# ============================================================================

# %% Cell 1: Imports and Setup
import os
import sys
import pickle
import hashlib
import time
import gc
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
from itertools import product
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import svd as scipy_svd
import pandas as pd

print("✓ Imports successful! (CIFAR-10 Production Version)")

# %% Cell 2: Kaggle Environment Setup
def setup_kaggle_environment():
    """Setup paths and check Kaggle environment."""
    if os.path.exists('/kaggle/working'):
        BASE_DIR = Path('/kaggle/working')
    else:
        BASE_DIR = Path('.')
    
    RESULTS_DIR = BASE_DIR / 'results'
    RESULTS_DIR.mkdir(exist_ok=True)
    
    PLOTS_DIR = BASE_DIR / 'plots'
    PLOTS_DIR.mkdir(exist_ok=True)
    
    CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    print(f"✓ Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
        print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"✓ Results directory: {RESULTS_DIR}")
    print(f"✓ Plots directory: {PLOTS_DIR}")
    
    return RESULTS_DIR, PLOTS_DIR, CHECKPOINT_DIR, DEVICE

RESULTS_DIR, PLOTS_DIR, CHECKPOINT_DIR, DEVICE = setup_kaggle_environment()

# %% Cell 3: Time and Memory Management
class ExperimentTimer:
    """Manage experiment time to stay within Kaggle limits."""
    
    def __init__(self, max_hours: float = 11.5):
        self.start_time = time.time()
        self.max_seconds = max_hours * 3600
        
    def elapsed_hours(self) -> float:
        return (time.time() - self.start_time) / 3600
    
    def remaining_hours(self) -> float:
        return self.max_seconds / 3600 - self.elapsed_hours()
    
    def should_stop(self) -> bool:
        return self.elapsed_hours() >= (self.max_seconds / 3600)
    
    def print_status(self):
        print(f"⏱️  Elapsed: {self.elapsed_hours():.2f}h | Remaining: {self.remaining_hours():.2f}h")

def cleanup_memory():
    """Aggressive memory cleanup."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_memory_stats():
    """Get current memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        cached = torch.cuda.memory_reserved() / 1e9
        return f"GPU: {allocated:.2f}GB allocated, {cached:.2f}GB cached"
    return "CPU mode"

# %% Cell 4: Reproducibility
def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    pl.seed_everything(seed, workers=True)

# %% Cell 5: Dataset Generation (CIFAR-10 Subsets)
class CIFAR10Subset(Dataset):
    """
    CIFAR-10 Subset: Selects exactly n_per_class samples.
    Flattens images and normalizes them.
    """
    def __init__(self, n_per_class: int, train: bool = True, seed: int = 42, root: str = './data'):
        self.n_per_class = n_per_class
        self.train = train
        
        # CIFAR-10 Statistics
        mean = (0.4914, 0.4822, 0.4465)
        std = (0.2023, 0.1994, 0.2010)
        
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
        
        # Download/Load CIFAR10
        # We wrap this in try/except to handle potential network issues or pre-downloaded data
        try:
            full_dataset = torchvision.datasets.CIFAR10(
                root=root, train=train, download=True, transform=transform
            )
        except:
            print("⚠️ Download failed, checking if data exists locally...")
            full_dataset = torchvision.datasets.CIFAR10(
                root=root, train=train, download=False, transform=transform
            )
        
        if train:
            # Select indices deterministic based on seed
            targets = np.array(full_dataset.targets)
            selected_indices = []
            
            rng = np.random.RandomState(seed)
            
            for k in range(10):
                class_indices = np.where(targets == k)[0]
                rng.shuffle(class_indices)
                if n_per_class > len(class_indices):
                    # Fallback for extreme cases, though CIFAR has 5k/class
                    selected_indices.extend(class_indices)
                else:
                    selected_indices.extend(class_indices[:n_per_class])
            
            self.dataset = Subset(full_dataset, selected_indices)
            self.targets = torch.LongTensor(targets[selected_indices])
        else:
            self.dataset = full_dataset
            self.targets = torch.LongTensor(full_dataset.targets)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        img, target = self.dataset[idx]
        # Flatten: 3x32x32 -> 3072
        return img.view(-1), target

def create_data_loaders(K: int, d0: int, n_per_class: int, 
                        batch_size: int, seed: int = 42):
    """Create train/val/test loaders with CIFAR subsets."""
    # 1. Train Set
    train_full = CIFAR10Subset(n_per_class, train=True, seed=seed)
    
    # 2. Validation Split (10% or min 10 samples)
    N = len(train_full)
    n_val = max(10, int(0.1 * N))
    if n_val >= N: n_val = 0 # Handle very small N case
    n_train = N - n_val
    
    if n_val > 0:
        train_ds, val_ds = random_split(
            train_full, [n_train, n_val],
            generator=torch.Generator().manual_seed(seed)
        )
    else:
        train_ds = train_full
        val_ds = train_full # Fallback validation on train (not ideal but safe for code)

    # 3. Test Set (Full CIFAR test set)
    test_ds = CIFAR10Subset(n_per_class=0, train=False)
    
    # Loaders
    eff_batch_size = batch_size if batch_size > 0 else (min(1024, len(train_ds)))
    
    train_loader = DataLoader(train_ds, batch_size=eff_batch_size, shuffle=True, 
                              num_workers=0, persistent_workers=False, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=1000, shuffle=False,
                           num_workers=0, persistent_workers=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=1000, shuffle=False,
                            num_workers=0, pin_memory=True)
    
    return train_loader, val_loader, test_loader

# %% Cell 6: Network Architecture (CIFAR Adapted)
class BottleneckMLP(nn.Module):
    """
    Deep MLP for CIFAR-10.
    Input(3072) -> Project(d0) -> ReLU -> [Hidden Layers] -> Output(K)
    """
    
    def __init__(self, d0: int, K: int, L_hidden: int, d_b: int,
                 width_pattern: str = 'constant_narrow', use_relu: bool = True):
        super().__init__()
        self.d0 = d0 # 1000
        self.K = K
        
        # CIFAR Input Projection: 3072 -> d0
        self.input_dim = 3072
        self.projector = nn.Sequential(
            nn.Linear(self.input_dim, d0),
            nn.ReLU() if use_relu else nn.Identity()
        )
        
        self.L_hidden = L_hidden
        self.d_b = d_b
        self.width_pattern = width_pattern
        self.use_relu = use_relu
        
        widths = self._build_widths()
        self.widths = widths
        
        layers = []
        for i in range(len(widths) - 1):
            layers.append(nn.Linear(widths[i], widths[i+1]))
            if i < len(widths) - 2:
                if use_relu:
                    layers.append(nn.ReLU())
        
        self.network = nn.Sequential(*layers)
        
    def _build_widths(self) -> List[int]:
        if self.width_pattern == 'constant_narrow':
            widths = [self.d0] + [self.d_b] * self.L_hidden + [self.K]
        elif self.width_pattern == 'wide_narrow_wide':
            if self.L_hidden < 3:
                widths = [self.d0, self.d_b, self.K]
            else:
                n_wide = (self.L_hidden - 1) // 2
                widths = ([self.d0] + [512] * n_wide + [self.d_b] + 
                         [512] * (self.L_hidden - n_wide - 1) + [self.K])
        else:
            raise ValueError(f"Unknown width pattern: {self.width_pattern}")
        return widths
    
    def forward(self, x):
        h = self.projector(x)
        return self.network(h)
    
    def get_layer_activations(self, x: torch.Tensor) -> List[torch.Tensor]:
        """Extract activations including projection."""
        h = self.projector(x)
        activations = [h.detach()] # Layer 0 is after projection
        
        curr = h
        for layer in self.network:
            curr = layer(curr)
            if isinstance(layer, (nn.Linear, nn.ReLU)):
                activations.append(curr.detach())
        return activations

# %% Cell 7: Metrics Computation (Updated)
class MetricsComputer:
    
    @staticmethod
    def stable_rank(H: torch.Tensor) -> float:
        """Stable rank: ||H||_F^2 / ||H||_2^2"""
        try:
            if H.shape[0] < 2: return float(H.shape[1])
            frob_norm = torch.norm(H, p='fro').item() ** 2
            spec_norm = torch.linalg.matrix_norm(H, ord=2).item() ** 2
            if not np.isfinite(frob_norm) or not np.isfinite(spec_norm) or spec_norm < 1e-10:
                return float(H.shape[1])
            return max(1.0, frob_norm / spec_norm)
        except: return float(H.shape[1])

    @staticmethod
    def nc1_metric(H: torch.Tensor, y: torch.Tensor, K: int) -> float:
        try:
            H_np = H.cpu().numpy(); y_np = y.cpu().numpy()
            if not np.isfinite(H_np).all(): return 0.5
            
            mu_G = H_np.mean(axis=0)
            Sigma_W = 0; Sigma_T = 0
            for k in range(K):
                H_k = H_np[y_np == k]
                if len(H_k) == 0: continue
                mu_k = H_k.mean(axis=0)
                Sigma_W += (H_k - mu_k).T @ (H_k - mu_k)
                Sigma_T += len(H_k) * np.outer(mu_k - mu_G, mu_k - mu_G)
            Sigma_T += Sigma_W
            
            tr_W = np.trace(Sigma_W)
            tr_T = np.trace(Sigma_T)
            return float(np.clip(tr_W / (tr_T + 1e-10), 0.0, 1.0))
        except: return 0.5

    @staticmethod
    def singular_values(H: torch.Tensor, k: int = 10) -> np.ndarray:
        try:
            if H.shape[0] < k: k = H.shape[0]
            if not torch.isfinite(H).all(): return np.ones(k)
            U, S, V = torch.svd_lowrank(H, q=min(k, min(H.shape)))
            return S.cpu().numpy()
        except: return np.ones(k)

    @staticmethod
    def compute_all_metrics(model: nn.Module, train_loader: DataLoader, 
                           test_loader: DataLoader, K: int, device: str) -> Dict:
        """Compute metrics + Test Accuracy."""
        model.eval()
        model = model.to(device)
        
        # 1. Layer Metrics on Train Data
        all_X, all_y = [], []
        for X, y in train_loader:
            all_X.append(X.to(device))
            all_y.append(y.to(device))
        
        if not all_X: return {}
        
        X_full = torch.cat(all_X, dim=0)
        y_full = torch.cat(all_y, dim=0)
        
        with torch.no_grad():
            activations = model.get_layer_activations(X_full)
        
        metrics = {}
        L = len(activations) - 1
        
        for ell in range(1, L + 1):
            H = activations[ell]
            if len(H.shape) == 1: H = H.unsqueeze(0)
            if ell == L or H.shape[1] == K: continue
            
            metrics[f'layer_{ell}'] = {
                'stable_rank': MetricsComputer.stable_rank(H),
                'nc1': MetricsComputer.nc1_metric(H, y_full, K),
                'singular_values': MetricsComputer.singular_values(H, k=10),
                'width': H.shape[1]
            }
        
        # 2. Generalization: Full Test Set Accuracy
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                logits = model(X)
                preds = logits.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        metrics['test_acc_full'] = correct / total if total > 0 else 0.0
        
        del X_full, y_full, activations
        cleanup_memory()
        
        return metrics

# %% Cell 8: Experiment Configuration
@dataclass
class ExperimentConfig:
    K: int = 10
    d0: int = 1000
    n_per_class: int = 50
    d_b: int = 64
    L_hidden: int = 4
    width_pattern: str = 'constant_narrow'
    wd: float = 1e-3
    lr: float = 0.05
    max_iters: int = 100000 
    batch_size: int = -1
    seed: int = 42
    use_relu: bool = True
    
    def to_hash(self) -> str:
        config_dict = asdict(self)
        config_dict.pop('seed')
        config_str = str(sorted(config_dict.items()))
        return hashlib.md5(config_str.encode()).hexdigest()[:8]

# %% Cell 9: Lightning Module
class BottleneckLightningModule(pl.LightningModule):
    
    def __init__(self, config: ExperimentConfig):
        super().__init__()
        self.save_hyperparameters()
        self.config = config
        
        self.model = BottleneckMLP(
            d0=config.d0,
            K=config.K,
            L_hidden=config.L_hidden,
            d_b=config.d_b,
            width_pattern=config.width_pattern,
            use_relu=config.use_relu
        )
        
        self.criterion = nn.CrossEntropyLoss()
        
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.parameters(),
            lr=self.config.lr,
            momentum=0.9,
            weight_decay=self.config.wd
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.config.max_iters
        )
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}

# %% Cell 10: Training Function
def train_single_run(config: ExperimentConfig, timer: ExperimentTimer) -> Optional[Dict]:
    """Train a single configuration and return metrics."""
    if timer.should_stop():
        print("⏱️  Time limit reached, skipping this run")
        return None
    
    set_seed(config.seed)
    
    try:
        # Create Loaders (Train, Val, Test)
        train_loader, val_loader, test_loader = create_data_loaders(
            config.K, config.d0, config.n_per_class, config.batch_size, config.seed
        )
        
        model = BottleneckLightningModule(config)
        
        # Aggressive Early Stopping for time management
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=5000, 
            min_delta=1e-4,
            mode='min',
            verbose=False
        )
        
        trainer = pl.Trainer(
            max_steps=config.max_iters,
            logger=False,
            callbacks=[early_stop],
            enable_progress_bar=False,
            enable_checkpointing=False,
            accelerator='auto',
            devices=1,
            deterministic=True
        )
        
        trainer.fit(model, train_loader, val_loader)
        
        device = next(model.parameters()).device
        model.eval()
        
        metrics = MetricsComputer.compute_all_metrics(
            model.model, train_loader, test_loader, config.K, str(device)
        )
        
        result = {
            'config': asdict(config),
            'metrics': metrics,
            'final_train_acc': trainer.callback_metrics.get('train_acc', torch.tensor(0.0)).item(),
            'final_val_loss': trainer.callback_metrics.get('val_loss', torch.tensor(float('inf'))).item(),
            'steps_taken': trainer.global_step
        }
        
        del model, trainer, train_loader, val_loader, test_loader
        cleanup_memory()
        
        return result
        
    except Exception as e:
        print(f"❌ Run failed: {e}")
        cleanup_memory()
        return None

# %% Cell 11: Checkpoint Management
def save_checkpoint(results: List[Dict], checkpoint_path: Path):
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"💾 Checkpoint saved: {len(results)} results")

def load_checkpoint(checkpoint_path: Path) -> List[Dict]:
    if checkpoint_path.exists():
        with open(checkpoint_path, 'rb') as f:
            results = pickle.load(f)
        print(f"📂 Loaded checkpoint: {len(results)} results")
        return results
    return []

# %% Cell 12: Sweep Function
def run_sweep(sweep_params: Dict, n_seeds: int, timer: ExperimentTimer, 
              checkpoint_path: Path) -> List[Dict]:
    
    all_results = load_checkpoint(checkpoint_path)
    
    keys = list(sweep_params.keys())
    values = [sweep_params[k] for k in keys]
    
    completed_configs = set()
    for res in all_results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'], 
               cfg['width_pattern'], cfg['wd'], cfg['seed'])
        completed_configs.add(key)
    
    total_configs = np.prod([len(v) for v in values]) * n_seeds
    config_count = 0
    
    for combo in product(*values):
        config_dict = dict(zip(keys, combo))
        
        for seed in range(n_seeds):
            config = ExperimentConfig(**config_dict, seed=seed)
            
            key = (config.n_per_class, config.d_b, config.L_hidden,
                   config.width_pattern, config.wd, seed)
            if key in completed_configs:
                continue
            
            config_count += 1
            if timer.should_stop():
                print(f"\n⏱️  Time limit reached after {config_count} runs")
                break
            
            print(f"\n[{len(all_results)+1}/{total_configs}] n={config.n_per_class}, "
                  f"d_b={config.d_b}, L={config.L_hidden}, wd={config.wd}, seed={seed}")
            timer.print_status()
            
            result = train_single_run(config, timer)
            if result is not None:
                all_results.append(result)
                if len(all_results) % 5 == 0:
                    save_checkpoint(all_results, checkpoint_path)
        
        if timer.should_stop(): break
    
    save_checkpoint(all_results, checkpoint_path)
    return all_results

# %% Cell 13: VISUALIZATION FUNCTIONS (Full Set of 14 Plots)

def plot_rank_evolution(results: List[Dict], save_path: Path):
    if len(results) == 0: return
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['width_pattern'])
        if key not in config_groups: config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax = axes[idx]
        n, d_b, pattern = key
        layer_ranks = {}
        for res in group:
            metrics = res['metrics']
            for layer_name, layer_metrics in metrics.items():
                if layer_name.startswith('layer_'):
                    layer_idx = int(layer_name.split('_')[1])
                    rank = layer_metrics['stable_rank']
                    if np.isfinite(rank):
                        if layer_idx not in layer_ranks: layer_ranks[layer_idx] = []
                        layer_ranks[layer_idx].append(rank)
        
        if not layer_ranks: continue
        layers = sorted(layer_ranks.keys())
        means = [np.mean(layer_ranks[l]) for l in layers]
        stds = [np.std(layer_ranks[l]) for l in layers]
        ax.plot(layers, means, 'o-', linewidth=2)
        ax.fill_between(layers, np.array(means)-np.array(stds), np.array(means)+np.array(stds), alpha=0.3)
        ax.set_title(f'n={n}, d_b={d_b}')
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_singular_spectra(results: List[Dict], save_path: Path):
    if len(results) == 0: return
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'])
        if key not in config_groups: config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax = axes[idx]
        n, d_b, L = key
        for res in group[:1]:
            metrics = res['metrics']
            middle_layer = L // 2 + 1
            layer_key = f'layer_{middle_layer}'
            if layer_key in metrics:
                svals = metrics[layer_key]['singular_values']
                if len(svals) > 0: ax.loglog(range(1, len(svals)+1), svals, 'o-')
        ax.set_title(f'n={n}, d_b={d_b}, L={L}')
        ax.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_bottleneck_location(results: List[Dict], save_path: Path):
    bottleneck_layers = []
    for res in results:
        metrics = res['metrics']
        ranks = {}
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): ranks[int(layer_name.split('_')[1])] = rank
        if len(ranks) > 1: bottleneck_layers.append(min(ranks, key=ranks.get))
    if not bottleneck_layers: return
    plt.figure(figsize=(10, 6))
    plt.hist(bottleneck_layers, bins=max(10, len(set(bottleneck_layers))), edgecolor='black')
    plt.title('Distribution of Bottleneck Layer Location')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_scaling_law(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'nK': cfg['n_per_class']*cfg['K'], 'r_min': min_rank})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(8, 6))
    plt.loglog(df['nK'], df['r_min'], 'o', alpha=0.5)
    plt.xlabel('nK')
    plt.ylabel('Min Stable Rank')
    plt.title('Scaling Law')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_phase_diagram(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            nK = cfg['n_per_class'] * cfg['K']
            data.append({
                'log_n': np.log10(cfg['n_per_class']),
                'log_d_b': np.log10(cfg['d_b']),
                'norm_rank': min_rank / min(cfg['d_b'], nK)
            })
    if not data: return
    df = pd.DataFrame(data)
    pivot = df.pivot_table(values='norm_rank', index='log_d_b', columns='log_n', aggfunc='mean')
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot, annot=True, cmap='viridis')
    plt.title('Phase Diagram')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_training_dynamics(results: List[Dict], save_path: Path):
    # Plot rank vs layer for a few configs
    plot_rank_evolution(results, save_path)
    print(f"✓ Saved: {save_path.name}")

def plot_wd_ablation(results: List[Dict], save_path: Path):
    wd_values = sorted(list(set([r['config']['wd'] for r in results])))
    if not wd_values: return
    fig, axes = plt.subplots(1, min(len(wd_values), 4), figsize=(20, 5))
    if len(wd_values) == 1: axes = [axes]
    for idx, wd in enumerate(wd_values[:4]):
        subset = [r for r in results if r['config']['wd'] == wd]
        ranks = []
        for res in subset:
            metrics = res['metrics']
            curr_ranks = [m['stable_rank'] for k, m in metrics.items() if k.startswith('layer_')]
            if curr_ranks: ranks.append(np.mean(curr_ranks))
        if ranks:
            axes[idx].hist(ranks, bins=10)
            axes[idx].set_title(f'WD={wd}')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_depth_analysis(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'L': cfg['L_hidden'], 'r_min': min_rank})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df, x='L', y='r_min', marker='o')
    plt.title('Depth Analysis')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_capacity_analysis(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'capacity': min(cfg['d_b'], cfg['n_per_class']*cfg['K']/2), 'r_min': min_rank})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(8, 8))
    plt.scatter(df['capacity'], df['r_min'])
    plt.plot([0, df['capacity'].max()], [0, df['capacity'].max()], 'r--')
    plt.title('Capacity Analysis')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_accuracy_vs_rank(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'r_min': min_rank, 'acc': res.get('final_train_acc', 0)})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    plt.scatter(df['r_min'], df['acc'])
    plt.title('Accuracy vs Rank')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_architecture_comparison(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'pattern': cfg['width_pattern'], 'r_min': min_rank})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='pattern', y='r_min')
    plt.title('Architecture Comparison')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_test_acc_vs_rank(results: List[Dict], save_path: Path):
    """New: Test Accuracy vs Bottleneck Rank (Generalization)."""
    if len(results) < 5: return
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        test_acc = metrics.get('test_acc_full', 0)
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'r_min': min_rank, 'test_acc': test_acc, 'n': cfg['n_per_class']})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='r_min', y='test_acc', hue='n', palette='viridis', s=100)
    plt.title('Generalization vs Compression')
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_transfer_comparison(results: List[Dict], save_path: Path):
    """New: Rank Evolution on CIFAR."""
    plot_rank_evolution(results, save_path)
    print(f"✓ Saved Transfer Plot: {save_path.name}")

def generate_summary_table(results: List[Dict], save_path: Path):
    if len(results) == 0: return
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        ranks = [m['stable_rank'] for k, m in metrics.items() if k.startswith('layer_') and np.isfinite(m['stable_rank'])]
        if ranks:
            data.append({
                'n': cfg['n_per_class'], 'd_b': cfg['d_b'], 'L': cfg['L_hidden'],
                'r_min': min(ranks), 'test_acc': metrics.get('test_acc_full', 0)
            })
    if data:
        pd.DataFrame(data).to_csv(save_path)
        print(f"✓ Saved: {save_path.name}")

# %% Cell 14: Quick Test Run
print("\n" + "="*80)
print("🧪 RUNNING QUICK TEST (should complete in <1 minute)")
print("="*80)

test_timer = ExperimentTimer(max_hours=0.1)
test_config = ExperimentConfig(
    n_per_class=20, # Small subset
    d_b=16,
    L_hidden=3,
    max_iters=100,
    width_pattern='constant_narrow',
    wd=1e-3,
    seed=42
)

print(f"Testing with: n=20, d_b=16, L=3, max_iters=100")
test_result = train_single_run(test_config, test_timer)

if test_result is not None:
    print("✅ TEST PASSED! Proceeding to full sweep...")
    cleanup_memory()
else:
    print("❌ TEST FAILED! Check configuration and try again.")
    sys.exit(1)

# %% Cell 15: Full Sweep Configuration (Time-Optimized)
print("\n" + "="*80)
print("🚀 FULL SWEEP CONFIGURATION (CIFAR-10)")
print("="*80)

# 11 hours = 660 mins. ~3.0 mins/run -> ~220 runs.
sweep_params = {
    'n_per_class': [20, 50, 100, 200], 
    'd_b': [16, 32, 64, 128],
    'L_hidden': [4, 8],
    'width_pattern': ['constant_narrow', 'wide_narrow_wide'],
    'wd': [1e-3, 5e-3],
    'K': [10]
}

n_seeds = 3
max_iters = 100000 # Relies on early stopping (patience=5000)

total_configs = 1
for param_name, param_values in sweep_params.items():
    total_configs *= len(param_values)

total_runs = total_configs * n_seeds
estimated_time_per_run = 3.0 # minutes

print(f"Unique configurations: {total_configs}")
print(f"Total runs planned: {total_runs}")
print(f"Expected completion: ~{total_runs * estimated_time_per_run / 60:.1f} hours")
print("="*80)

ExperimentConfig.__dataclass_fields__['max_iters'].default = max_iters

# %% Cell 16: Run Full Sweep
print("\n🏃 Starting full sweep...")
main_timer = ExperimentTimer(max_hours=11.5)
checkpoint_path = CHECKPOINT_DIR / 'sweep_checkpoint.pkl'

all_results = run_sweep(
    sweep_params=sweep_params,
    n_seeds=n_seeds,
    timer=main_timer,
    checkpoint_path=checkpoint_path
)

print(f"\n✅ Sweep complete! {len(all_results)} results collected.")
print(f"⏱️  Total time: {main_timer.elapsed_hours():.2f} hours")

# %% Cell 17: Generate All Plots
print("\n" + "="*80)
print("📊 GENERATING VISUALIZATIONS")
print("="*80)

if len(all_results) > 0:
    print(f"Processing {len(all_results)} results...")
    
    # Standard Plots
    plot_rank_evolution(all_results, PLOTS_DIR / 'rank_evolution.png')
    plot_singular_spectra(all_results, PLOTS_DIR / 'singular_spectra.png')
    plot_bottleneck_location(all_results, PLOTS_DIR / 'bottleneck_location.png')
    plot_scaling_law(all_results, PLOTS_DIR / 'scaling_law.png')
    plot_phase_diagram(all_results, PLOTS_DIR / 'phase_diagram.png')
    plot_training_dynamics(all_results, PLOTS_DIR / 'training_dynamics.png')
    plot_wd_ablation(all_results, PLOTS_DIR / 'wd_ablation.png')
    plot_depth_analysis(all_results, PLOTS_DIR / 'depth_analysis.png')
    plot_capacity_analysis(all_results, PLOTS_DIR / 'capacity_analysis.png')
    plot_accuracy_vs_rank(all_results, PLOTS_DIR / 'accuracy_vs_rank.png')
    plot_architecture_comparison(all_results, PLOTS_DIR / 'architecture_comparison.png')
    
    # New CIFAR Plots
    plot_test_acc_vs_rank(all_results, PLOTS_DIR / 'test_acc_vs_rank.png')
    plot_transfer_comparison(all_results, PLOTS_DIR / 'transfer_comparison_rank.png')
    
    generate_summary_table(all_results, RESULTS_DIR / 'summary_stats.csv')
    
    # Save Results
    with open(RESULTS_DIR / 'all_results.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    
    # Export CSV
    rows = []
    for res in all_results:
        cfg = res['config']
        base_row = {
            'n_per_class': cfg['n_per_class'],
            'd_b': cfg['d_b'],
            'L_hidden': cfg['L_hidden'],
            'wd': cfg['wd'],
            'seed': cfg['seed'],
            'train_acc': res.get('final_train_acc', 0),
            'test_acc': res['metrics'].get('test_acc_full', 0)
        }
        rows.append(base_row)
    
    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'summary_results.csv', index=False)
    print(f"✓ Saved all results")
    
else:
    print("⚠️  No results to visualize")

# %% Cell 18: Final Summary
print("\n" + "="*80)
print("📈 EXPERIMENT SUMMARY")
print("="*80)
if len(all_results) > 0:
    print(f"Total runs: {len(all_results)}")
    accs = [r['metrics'].get('test_acc_full', 0) for r in all_results]
    print(f"Mean Test Acc: {np.mean(accs):.3f}")
    print(f"Max Test Acc: {np.max(accs):.3f}")
print("✅ EXPERIMENT COMPLETE!")

Seed set to 42


✓ Imports successful! (CIFAR-10 Production Version)
✓ Device: cuda
✓ GPU: Tesla P100-PCIE-16GB
✓ GPU Memory: 17.06 GB
✓ Results directory: /kaggle/working/results
✓ Plots directory: /kaggle/working/plots

🧪 RUNNING QUICK TEST (should complete in <1 minute)
Testing with: n=20, d_b=16, L=3, max_iters=100


100%|██████████| 170M/170M [00:06<00:00, 28.0MB/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BottleneckMLP    │  3.1 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 3.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.1 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_steps=100` reached.
Seed set to 0


... (637 outputs left out)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BottleneckMLP    │  3.9 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 3.9 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.9 M                                                                                                
Total estimated model params size (MB): 15                                                                         
Modules in train mode: 15                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0


⏱️  Time limit reached after 128 runs
💾 Checkpoint saved: 127 results

✅ Sweep complete! 127 results collected.
⏱️  Total time: 11.63 hours

📊 GENERATING VISUALIZATIONS
Processing 127 results...
✓ Saved: rank_evolution.png
✓ Saved: singular_spectra.png
✓ Saved: bottleneck_location.png
✓ Saved: scaling_law.png
✓ Saved: phase_diagram.png
✓ Saved: training_dynamics.png
✓ Saved: training_dynamics.png
✓ Saved: wd_ablation.png
✓ Saved: depth_analysis.png
✓ Saved: capacity_analysis.png
✓ Saved: accuracy_vs_rank.png
✓ Saved: architecture_comparison.png
✓ Saved: test_acc_vs_rank.png
✓ Saved: transfer_comparison_rank.png
✓ Saved Transfer Plot: transfer_comparison_rank.png
✓ Saved: summary_stats.csv
✓ Saved all results

📈 EXPERIMENT SUMMARY
Total runs: 127
Mean Test Acc: 0.137
Max Test Acc: 0.274
✅ EXPERIMENT COMPLETE!
